# Tokens, Embeddings, and Context

## Learning Objectives

Until now, our models have operated on numerical inputs.

For example, in regression we could directly give a model values such as:

```text
x = 2.5
```

Language creates a new problem:

> Neural networks cannot directly operate on words such as `"cat"`, `"model"`, or `"training"`.

We need a way to transform discrete pieces of text into numerical representations that a neural network can process and eventually learn from.

In this notebook, we will build that pipeline from first principles:

```text
raw text
↓
tokens
↓
vocabulary
↓
token IDs
↓
embeddings
↓
context windows
↓
context embeddings
```

By the end of the notebook, we should understand:

- what a token is
- what tokenization does
- what a vocabulary is
- why tokens are assigned integer IDs
- why token IDs are not themselves useful numerical features
- what an embedding is
- what `nn.Embedding` actually stores
- why embeddings are learnable parameters
- how embedding tensor shapes work
- what a context window is
- how text becomes supervised `context → target` examples
- what the dimensions `B`, `T`, and `C` mean

The goal is not yet to build a language model.

First, we want to understand exactly how text becomes numerical input that a language model can later use.

In [24]:
import torch
import torch.nn as nn

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1


## 1. A Tiny Text Corpus

Real language models are trained on enormous text datasets.

For learning the mechanics, however, a tiny corpus is much more useful because we can inspect every token, ID, tensor, and embedding directly.

We will use three short sentences:

```text
the cat sat on the mat
the cat ate the fish
the dog sat on the rug
```

At this point these are ordinary Python strings.

A neural network cannot directly perform mathematical operations on them.

Our first job is therefore to turn the text into smaller discrete units called **tokens**.

In [25]:
sentences = [
    "the cat sat on the mat",
    "the cat ate the fish",
    "the dog sat on the rug",
]

for sentence in sentences:
    print(sentence)

the cat sat on the mat
the cat ate the fish
the dog sat on the rug


## 2. Tokenization

A **token** is one discrete unit of text presented to a language model.

For this first experiment, we will use an intentionally simple tokenizer:

```python
sentence.split()
```

This splits text at whitespace.

For example:

```text
"the cat sat"
```

becomes:

```text
["the", "cat", "sat"]
```

This means our tokens are whole words.

### Is this how modern LLMs tokenize text?

Usually not.

Modern language models commonly use **subword tokenization**. A word may be represented by one token or broken into several smaller pieces.

For example, depending on the tokenizer, something conceptually like:

```text
"unbelievable"
```

might be represented using pieces such as:

```text
"un"
"believ"
"able"
```

The exact behavior depends on the tokenizer.

Subword tokenization helps models handle:

- rare words
- new words
- word variations
- large vocabularies

We will study real tokenizers later.

For now, word-level tokenization removes that complexity so we can understand the underlying language-model mechanics first.

In [26]:
tokenized_sentences = [
    sentence.split()
    for sentence in sentences
]

for tokens in tokenized_sentences:
    print(tokens)

['the', 'cat', 'sat', 'on', 'the', 'mat']
['the', 'cat', 'ate', 'the', 'fish']
['the', 'dog', 'sat', 'on', 'the', 'rug']


## 3. Flattening the Tokens

We have tokenized each sentence separately:

```text
sentence 1 → list of tokens
sentence 2 → list of tokens
sentence 3 → list of tokens
```

To construct our vocabulary, we want to collect all tokens that occur anywhere in the corpus.

We can therefore flatten the tokenized sentences into one list.

Importantly, we will still keep `tokenized_sentences` separately.

Later, when we create language-model context windows, keeping sentence boundaries will prevent us from accidentally constructing examples such as:

```text
end of sentence 1 → beginning of sentence 2
```

unless we explicitly decide that this behavior is appropriate.

In [27]:
all_tokens = [
    token
    for sentence_tokens in tokenized_sentences
    for token in sentence_tokens
]

print(all_tokens)
print("\nTotal number of tokens:", len(all_tokens))

['the', 'cat', 'sat', 'on', 'the', 'mat', 'the', 'cat', 'ate', 'the', 'fish', 'the', 'dog', 'sat', 'on', 'the', 'rug']

Total number of tokens: 17


## 4. Vocabulary

A **vocabulary** is the set of distinct tokens that the model knows how to represent.

Our corpus contains repeated words.

For example:

```text
the
cat
sat
```

appear more than once.

The vocabulary contains each distinct token only once.

If the corpus contains:

```text
the cat sat
the dog sat
```

then the vocabulary is conceptually:

```text
the
cat
sat
dog
```

not every occurrence of those words.

We will sort the vocabulary only so that our output is deterministic and easy to inspect.

The alphabetical ordering has no semantic meaning.

In [28]:
vocab = sorted(set(all_tokens))

print("Vocabulary:")
print(vocab)

print("\nVocabulary size:", len(vocab))

Vocabulary:
['ate', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']

Vocabulary size: 9


## 5. Token IDs

Neural-network operations cannot use strings such as:

```text
"cat"
"dog"
"the"
```

directly.

We therefore assign every vocabulary token an integer ID.

For example:

```text
"cat" → 1
"dog" → 2
```

The integer is only an **index**.

It is extremely important not to interpret the value numerically.

If:

```text
"cat" → 1
"dog" → 2
"the" → 8
```

this does **not** imply:

```text
dog = 2 × cat
```

and it does not imply that:

```text
token 7 is more similar to token 8
than token 1 is to token 8
```

Token IDs are categorical identifiers.

We will create two dictionaries:

```text
token_to_id
```

for converting text into IDs, and:

```text
id_to_token
```

for converting IDs back into readable text.

In [29]:
token_to_id = {
    token: idx
    for idx, token in enumerate(vocab)
}

id_to_token = {
    idx: token
    for token, idx in token_to_id.items()
}

print("Token → ID:")
print(token_to_id)

print("\nID → Token:")
print(id_to_token)

Token → ID:
{'ate': 0, 'cat': 1, 'dog': 2, 'fish': 3, 'mat': 4, 'on': 5, 'rug': 6, 'sat': 7, 'the': 8}

ID → Token:
{0: 'ate', 1: 'cat', 2: 'dog', 3: 'fish', 4: 'mat', 5: 'on', 6: 'rug', 7: 'sat', 8: 'the'}


## 6. Encoding Text

Converting tokens into their integer IDs is called **encoding**.

Suppose:

```text
"the" → 8
"cat" → 1
"sat" → 6
```

Then:

```text
["the", "cat", "sat"]
```

becomes:

```text
[8, 1, 6]
```

We will encode each sentence separately.

We use a PyTorch tensor with:

```python
dtype=torch.long
```

because these values are integer indices.

This will also be the dtype expected later by `nn.Embedding`.

In [30]:
encoded_sentences = [
    torch.tensor(
        [token_to_id[token] for token in sentence_tokens],
        dtype=torch.long,
    )
    for sentence_tokens in tokenized_sentences
]

for sentence, encoded in zip(sentences, encoded_sentences):
    print(sentence)
    print(encoded)
    print("shape:", encoded.shape)
    print()

the cat sat on the mat
tensor([8, 1, 7, 5, 8, 4])
shape: torch.Size([6])

the cat ate the fish
tensor([8, 1, 0, 8, 3])
shape: torch.Size([5])

the dog sat on the rug
tensor([8, 2, 7, 5, 8, 6])
shape: torch.Size([6])



## 7. Encoding Is Reversible

At this point our transformation is:

```text
text
↓
tokens
↓
token IDs
```

Because we also stored `id_to_token`, we can reverse the ID mapping.

For example:

```text
[8, 1, 6]
```

can be converted back into:

```text
["the", "cat", "sat"]
```

This is useful later because models operate on numerical IDs, while humans usually want to inspect readable tokens.

In [31]:
example_encoded = encoded_sentences[0]

decoded_tokens = [
    id_to_token[token_id.item()]
    for token_id in example_encoded
]

print("Encoded:")
print(example_encoded)

print("\nDecoded:")
print(decoded_tokens)

Encoded:
tensor([8, 1, 7, 5, 8, 4])

Decoded:
['the', 'cat', 'sat', 'on', 'the', 'mat']


## 8. Why Token IDs Are Not Enough

We now have numerical values such as:

```text
[8, 1, 6, 5, 8, 4]
```

It may look like the text has already become numerical input.

But feeding these IDs directly into a neural network as ordinary numbers would create a false numerical relationship.

For example, if:

```text
cat → 1
dog → 2
the → 8
```

then ordinary arithmetic would treat the distances as:

```text
distance(cat, dog) = |1 - 2| = 1

distance(cat, the) = |1 - 8| = 7
```

But those distances are artifacts of the arbitrary ID assignment.

They tell us nothing about the meanings of the words.

We need a representation where each token can have multiple learnable numerical features.

That representation is an **embedding**.

## 9. Embeddings

An **embedding** maps each token ID to a vector of floating-point values.

Suppose:

```text
vocabulary size = V
embedding dimension = C
```

Then the embedding table has shape:

```text
[V, C]
```

There is one row for every token in the vocabulary.

Conceptually:

```text
token ID 0 → [ 0.12, -0.41,  0.77,  0.03 ]
token ID 1 → [-0.55,  0.18,  0.22,  0.91 ]
token ID 2 → [ 0.67, -0.09, -0.31,  0.44 ]
...
```

Looking up token ID `i` returns:

```text
embedding_table[i]
```

which is a vector of length `C`.

### Why use a vector instead of one number?

One scalar provides only one degree of freedom.

A vector allows the model to distribute useful information across multiple learned dimensions.

Importantly, we do **not** manually assign meanings to those dimensions.

The model learns whatever representation is useful for its training objective.

## 10. Why Are We Using `embedding_dim = 4`?

For this experiment, we will choose:

```python
embedding_dim = 4
```

The number `4` is **not** a special value and it is not an optimal embedding size.

It is a pedagogical choice.

We want the representation to be:

```text
small enough to inspect directly
but
large enough to show that one token can have multiple learned features
```

For example, `"cat"` might currently be represented as:

```text
[0.21, -0.73, 0.14, 1.02]
```

These four values do not have predefined interpretations such as:

```text
dimension 1 = animal
dimension 2 = size
dimension 3 = color
dimension 4 = frequency
```

They are simply coordinates in a learned representation space.

During training, gradient descent can change these coordinates so that they become useful for predicting tokens.

The important distinction is:

```text
vocabulary size V
→ number of different tokens

embedding dimension C
→ number of learned values used to represent each token
```

If we chose:

```python
embedding_dim = 1
```

each token would receive only one learnable scalar.

If we chose:

```python
embedding_dim = 512
```

the same mechanism would work, but it would be unnecessarily large for this tiny experiment and difficult to inspect manually.

Real language models often use embedding dimensions in the hundreds or thousands.

There is no general rule such as:

```text
embedding dimension = sqrt(vocabulary size)
```

for language models.

Embedding dimension is an architectural hyperparameter related to model capacity, memory, compute, and the rest of the network architecture.

Here, `4` simply keeps the mechanics visible.

In [32]:
vocab_size = len(vocab)
embedding_dim = 4

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
)

print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embedding_dim)
print("Embedding table shape:", embedding.weight.shape)

Vocabulary size: 9
Embedding dimension: 4
Embedding table shape: torch.Size([9, 4])


## 11. What `nn.Embedding` Actually Stores

When we create:

```python
nn.Embedding(V, C)
```

PyTorch creates a learnable parameter matrix:

```text
embedding.weight
```

with shape:

```text
[V, C]
```

Each row corresponds to one vocabulary ID.

For example:

```text
row 0 → embedding for token ID 0
row 1 → embedding for token ID 1
row 2 → embedding for token ID 2
...
```

At the moment, these values are randomly initialized.

They have not learned anything about language yet.

That distinction matters:

```text
embedding table exists      ✓
embedding lookup works      ✓
embeddings are trained      ✗ not yet
```

Later, once we define a prediction task and loss, training will look conceptually like:

```text
prediction
↓
loss
↓
backpropagation
↓
embedding gradients
↓
optimizer update
```

Only then will the embedding vectors begin changing in response to the language-model objective.

In [33]:
print(embedding.weight)

print("\nRequires gradients:")
print(embedding.weight.requires_grad)

Parameter containing:
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036],
        [-0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396],
        [ 0.3189, -0.4245,  0.3057, -0.7746],
        [ 0.0349,  0.3211,  1.5736, -0.8455],
        [-1.2742,  2.1228, -1.2347, -0.4879],
        [-1.4181,  0.8963,  0.0499,  2.2667]], requires_grad=True)

Requires gradients:
True


## 12. Looking Up One Token

Suppose we want the representation for `"cat"`.

First we find its token ID:

```text
"cat"
↓
token_to_id
↓
integer ID
```

Then we use that ID to select the corresponding row of the embedding table:

```text
token ID
↓
embedding table lookup
↓
embedding vector
```

This is why token IDs are useful: they act as row indexes into the embedding table.

In [34]:
cat_id = token_to_id["cat"]

print("Token:", "cat")
print("Token ID:", cat_id)

print("\nEmbedding:")
print(embedding.weight[cat_id])

print("\nEmbedding shape:")
print(embedding.weight[cat_id].shape)

Token: cat
Token ID: 1

Embedding:
tensor([ 0.6784, -1.2345, -0.0431, -1.6047], grad_fn=<SelectBackward0>)

Embedding shape:
torch.Size([4])


## 13. `nn.Embedding` Is a Lookup Operation

We can perform the lookup directly:

```python
embedding.weight[cat_id]
```

or ask the embedding layer to perform it:

```python
embedding(torch.tensor(cat_id))
```

Both refer to the same row of the embedding table.

`nn.Embedding` therefore behaves conceptually like a learnable lookup table.

This is different from treating the token ID itself as a numerical feature.

The ID chooses **which representation** to retrieve.

In [35]:
cat_id_tensor = torch.tensor(cat_id, dtype=torch.long)

manual_lookup = embedding.weight[cat_id]
layer_lookup = embedding(cat_id_tensor)

print("Direct row lookup:")
print(manual_lookup)

print("\nnn.Embedding lookup:")
print(layer_lookup)

print("\nAre they equal?")
print(torch.allclose(manual_lookup, layer_lookup))

Direct row lookup:
tensor([ 0.6784, -1.2345, -0.0431, -1.6047], grad_fn=<SelectBackward0>)

nn.Embedding lookup:
tensor([ 0.6784, -1.2345, -0.0431, -1.6047], grad_fn=<EmbeddingBackward0>)

Are they equal?
True


## 14. Embedding an Entire Sentence

`nn.Embedding` can look up many token IDs at once.

Suppose a sentence contains:

```text
T
```

tokens.

Its encoded representation has shape:

```text
[T]
```

Passing that tensor through the embedding layer replaces every ID with a vector of length `C`.

Therefore:

```text
[T]
↓
Embedding
↓
[T, C]
```

The number of tokens does not change.

What changes is the representation of every token:

```text
one integer ID
→
C floating-point values
```

In [36]:
first_sentence_ids = encoded_sentences[0]

first_sentence_embeddings = embedding(first_sentence_ids)

print("Sentence:")
print(sentences[0])

print("\nToken IDs:")
print(first_sentence_ids)

print("\nToken ID shape:")
print(first_sentence_ids.shape)

print("\nEmbeddings:")
print(first_sentence_embeddings)

print("\nEmbedding shape:")
print(first_sentence_embeddings.shape)

Sentence:
the cat sat on the mat

Token IDs:
tensor([8, 1, 7, 5, 8, 4])

Token ID shape:
torch.Size([6])

Embeddings:
tensor([[-1.4181,  0.8963,  0.0499,  2.2667],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [-1.2742,  2.1228, -1.2347, -0.4879],
        [ 0.3189, -0.4245,  0.3057, -0.7746],
        [-1.4181,  0.8963,  0.0499,  2.2667],
        [ 1.6423, -0.1596, -0.4974,  0.4396]], grad_fn=<EmbeddingBackward0>)

Embedding shape:
torch.Size([6, 4])


## 15. Understanding `[T, C]`

For one sentence:

```text
the cat sat on the mat
```

we have:

```text
T = number of token positions
C = embedding dimension
```

If:

```text
T = 6
C = 4
```

then the embedding tensor has shape:

```text
[6, 4]
```

We can interpret the dimensions as:

```text
6 token positions
×
4 representation features per token
```

Each row corresponds to one token position.

Conceptually:

```text
             C embedding features
          ─────────────────────────►

token 0   [ •  •  •  • ]
token 1   [ •  •  •  • ]
token 2   [ •  •  •  • ]
token 3   [ •  •  •  • ]
token 4   [ •  •  •  • ]
token 5   [ •  •  •  • ]

  │
  │ T token positions
  ▼
```

This distinction between:

```text
token position
```

and:

```text
representation dimension
```

will become extremely important once we reach Transformers.

## 16. Language Modeling Introduces Context

So far, we have independently converted each token into an embedding.

But language modeling requires more than knowing which tokens are present.

A language model tries to predict what comes next based on previous tokens.

For example:

```text
the cat sat
```

could be used to predict:

```text
on
```

We can express this as:

```text
context → target
```

For a context length of 3:

```text
the cat sat → on
cat sat on → the
sat on the → mat
```

This transforms text into a supervised learning problem.

Previously, our regression examples looked like:

```text
input x → target y
```

Now they look like:

```text
sequence of previous tokens → next token
```

The basic supervised-learning structure has not disappeared.

The nature of the inputs and targets has changed.

## 17. Choosing a Context Length

We will use:

```python
context_length = 3
```

This means each training input contains three previous tokens.

We will often call sequence length:

```text
T
```

so here:

```text
T = 3
```

Again, `3` is a teaching choice.

A larger context length would allow a model to use more previous tokens.

For example:

```text
T = 3

the cat sat → on
```

versus:

```text
T = 5

the cat sat on the → mat
```

Modern language models may support contexts containing thousands or even more tokens.

For this notebook, a context of three tokens lets us inspect every example manually.

In [38]:
context_length = 3

contexts = []
targets = []

for sentence_ids in encoded_sentences:
    for i in range(len(sentence_ids) - context_length):
        context = sentence_ids[i:i + context_length]
        target = sentence_ids[i + context_length]

        contexts.append(context)
        targets.append(target)

contexts = torch.stack(contexts)
targets = torch.stack(targets)

print("Contexts:")
print(contexts)

print("\nTargets:")
print(targets)

print("\nContexts shape:", contexts.shape)
print("Targets shape:", targets.shape)

Contexts:
tensor([[8, 1, 7],
        [1, 7, 5],
        [7, 5, 8],
        [8, 1, 0],
        [1, 0, 8],
        [8, 2, 7],
        [2, 7, 5],
        [7, 5, 8]])

Targets:
tensor([5, 8, 4, 8, 3, 5, 8, 6])

Contexts shape: torch.Size([8, 3])
Targets shape: torch.Size([8])


## 18. Why We Built Context Windows Per Sentence

Notice that we created context windows separately inside each sentence.

We did **not** simply concatenate all three sentences and slide a window across the entire flattened corpus.

If we had done that blindly, we could create an example such as:

```text
on the mat → the
```

where the target `"the"` actually belongs to the beginning of the next sentence.

That would introduce an artificial cross-sentence relationship caused only by how we stored the data.

In real language-model training, document and sequence boundaries require explicit handling.

Common approaches include:

- special end-of-sequence tokens
- document boundaries
- packing sequences deliberately
- masking parts of the loss

We do not need those mechanisms yet.

For this tiny experiment, constructing windows separately for each sentence keeps the training examples easy to reason about.

## 19. Decoding the Context-Target Pairs

The tensors contain integer IDs, which are convenient for the model but difficult for us to inspect.

Let's decode every example back into readable tokens.

This gives us a direct view of the supervised dataset we have created:

```text
context → next-token target
```

In [39]:
for context_ids, target_id in zip(contexts, targets):
    context_tokens = [
        id_to_token[token_id.item()]
        for token_id in context_ids
    ]

    target_token = id_to_token[target_id.item()]

    print(context_tokens, "→", target_token)

['the', 'cat', 'sat'] → on
['cat', 'sat', 'on'] → the
['sat', 'on', 'the'] → mat
['the', 'cat', 'ate'] → the
['cat', 'ate', 'the'] → fish
['the', 'dog', 'sat'] → on
['dog', 'sat', 'on'] → the
['sat', 'on', 'the'] → rug


## 20. Understanding `[B, T]` and `[B]`

After creating all training examples, the context tensor has shape:

```text
[B, T]
```

where:

```text
B = number of examples
T = context length
```

Each row is one training example.

For example:

```text
[
    [the, cat, sat],
    [cat, sat, on],
    [sat, on, the],
    ...
]
```

In numerical form, these are token IDs.

Since every context currently predicts one next token, the target tensor has shape:

```text
[B]
```

Therefore:

```text
contexts
[B, T]

targets
[B]
```

### What does `B` mean?

`B` conventionally means **batch**.

Here we have constructed all available examples at once, so `B` currently equals the total number of examples.

Later, when datasets become larger, we will usually divide examples into smaller mini-batches during training.

### What does `T` mean?

`T` represents the number of token positions in the sequence.

It is often called:

- sequence length
- context length
- time dimension

The word **time** does not imply physical clock time.

It means progression through positions in a sequence.

## 21. Embedding the Context Tensor

Our contexts currently have shape:

```text
[B, T]
```

and contain integer token IDs.

But each individual ID can be replaced by its `C`-dimensional embedding.

Therefore:

```text
[B, T]
↓
nn.Embedding
↓
[B, T, C]
```

This is one of the most important shape transformations in language modeling.

The dimensions now mean:

```text
B = examples in the batch
T = token positions in each example
C = representation features for each token
```

In [40]:
context_embeddings = embedding(contexts)

print("Context IDs shape:")
print(contexts.shape)

print("\nContext embeddings shape:")
print(context_embeddings.shape)

Context IDs shape:
torch.Size([8, 3])

Context embeddings shape:
torch.Size([8, 3, 4])


## 22. Understanding `[B, T, C]`

Suppose:

```text
B = 9
T = 3
C = 4
```

Then:

```text
context_embeddings.shape
=
[9, 3, 4]
```

This means:

```text
9 training examples
×
3 token positions per example
×
4 learned features per token
```

We can visualize one example as:

```text
context = ["the", "cat", "sat"]

                 C = 4
              features
                 ↓

"the"   → [ •  •  •  • ]
"cat"   → [ •  •  •  • ]
"sat"   → [ •  •  •  • ]

   ↑
 T = 3 token positions
```

Stack many such examples together and we get:

```text
[B, T, C]
```

This shape will appear repeatedly throughout the rest of the course.

Transformer models will also operate primarily on tensors representing:

```text
batch
× sequence positions
× hidden representation
```

The dimension may later be called:

```text
C
d_model
hidden_size
embedding dimension
```

depending on the architecture or library.

The names vary, but the underlying idea is the same: each token position is represented by a vector.

In [41]:
example_index = 0

print("Context IDs:")
print(contexts[example_index])

print("\nContext tokens:")
print([
    id_to_token[token_id.item()]
    for token_id in contexts[example_index]
])

print("\nContext embeddings:")
print(context_embeddings[example_index])

print("\nContext embedding shape:")
print(context_embeddings[example_index].shape)

print("\nTarget ID:")
print(targets[example_index])

print("\nTarget token:")
print(id_to_token[targets[example_index].item()])

Context IDs:
tensor([8, 1, 7])

Context tokens:
['the', 'cat', 'sat']

Context embeddings:
tensor([[-1.4181,  0.8963,  0.0499,  2.2667],
        [ 0.6784, -1.2345, -0.0431, -1.6047],
        [-1.2742,  2.1228, -1.2347, -0.4879]], grad_fn=<SelectBackward0>)

Context embedding shape:
torch.Size([3, 4])

Target ID:
tensor(5)

Target token:
on


## 23. What Has Been Learned So Far?

Nothing yet.

This distinction is important.

We created:

```python
nn.Embedding(...)
```

which contains **learnable parameters**, but we have not trained those parameters.

The embedding values are still based on their random initialization.

So far we have only performed:

```text
text preprocessing
+
parameter initialization
+
embedding lookup
```

We have not yet performed:

```text
prediction
loss calculation
backpropagation
optimizer update
```

Therefore, we should not look at the current `"cat"` and `"dog"` embeddings and interpret their distances as learned semantic relationships.

They have not had a learning objective yet.

The next stage of language modeling will provide that objective.

In [42]:
print("Embedding parameters require gradients:")
print(embedding.weight.requires_grad)

print("\nGradient currently stored:")
print(embedding.weight.grad)

Embedding parameters require gradients:
True

Gradient currently stored:
None


## 24. The Pipeline We Have Built

We started with raw text:

```text
the cat sat on the mat
```

and transformed it step by step:

```text
raw text
↓
tokenization

["the", "cat", "sat", "on", "the", "mat"]

↓
vocabulary lookup

[token IDs]

↓
context construction

["the", "cat", "sat"] → "on"

↓
integer tensors

context: [T]
target: scalar

↓
stack examples

contexts: [B, T]
targets:  [B]

↓
embedding lookup

context embeddings: [B, T, C]
```

Three dimensions are now especially important:

```text
B = batch / number of examples
T = sequence length / token positions
C = embedding or representation dimension
```

We have crossed an important conceptual boundary:

```text
discrete symbols
→ continuous neural-network representations
```

But we still do not have a language model.

We have only prepared its inputs.

## 25. What Comes Next?

For every context we now have a representation such as:

```text
[B, T, C]
```

and a correct next-token target:

```text
[B]
```

The next problem is:

> How can a neural network use the context representation to decide which vocabulary token should come next?

If our vocabulary contains `V` possible tokens, the model will eventually need to produce something like:

```text
score for token 0
score for token 1
score for token 2
...
score for token V - 1
```

Those raw scores are called **logits**.

From there we will connect:

```text
context embeddings
↓
model
↓
logits over vocabulary
↓
probabilities
↓
next-token target
↓
cross-entropy loss
↓
backpropagation
```

That will turn the representations we created in this notebook into an actual learning problem.

Before moving there, however, the important mental model from this notebook is:

```text
token ID
≠ learned representation

token ID
↓
embedding lookup
↓
learned vector

sequence of token IDs
[B, T]

↓

sequence of vector representations
[B, T, C]
```